In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    row_number,
    sha2,
    substring,
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_application_fldt(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ApplicationID	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "ApplicationKey	string,"
        #     "ApplicationStrategyKey string,"
        #     "ApplicationScheduleKey string,"
        #     "ExposureKey	string,"
        #     "AgreementKey	string,"
        #     "PartyKey	string,"
        #     "CDI	string,"
        #     "TaxRegistrationNumber string,"
        #     "GroupCode string,"
        #     "BusinessPlanKey int,"
        #     "BusinessPlanSolutionTypeCode string,"
        #     "BusinessPlanSolutionType string,"
        #     "BusinessPlanSolutionStatusCode string,"
        #     "BusinessPlanSolutionStatus string,"
        #     "BusinessPlanComments string,"
        #     "BusinessPlanCreatedDate timestamp,"
        #     "BusinessPlanCreatedBy string,"
        #     "BusinessPlanLastUpdatedDate timestamp,"
        #     "BusinessPlanLastUpdatedBy string,"
        #     "ApplicationDate timestamp,"
        #     "ApplicationTypeCode	string,"
        #     "ApplicationType	string,"
        #     "ApplicationStatusCode	string,"
        #     "ApplicationStatus	string,"
        #     "IntentionDate	date,"
        #     "ApprovedDate	timestamp,"
        #     "RejectedDate	timestamp,"
        #     "CancelledDate	timestamp,"
        #     "ImplementationDate	string,"
        #     "BeginDate	date,"
        #     "ProposedTypeCode	string,"
        #     "ProposedType	string,"
        #     "ConfidenceTypeCode	string,"
        #     "ConfidenceType	string,"
        #     "Installment	string,"
        #     "PropositionUser	string,"
        #     "Tenor	string,"
        #     "TimeToAuction	int,"
        #     "TimeToRecovery	int,"
        #     "TimeToSell	int,"
        #     "PrepaymentDate	date,"
        #     "AuctionPrice	decimal(19,2),"
        #     "BalloonAmount	decimal(19,2),"
        #     "PrepaymentAmount	decimal(19,2),"
        #     "BusinessPlanStrategySolutionTypeCode	string,"
        #     "BusinessPlanStrategySolutionType	string,"
        #     "Interest	decimal(21,4),"
        #     "InterestPaymentPeriodicity	int,"
        #     "NetRecoveryAmount	decimal(19,2),"
        #     "BusinessPlanStrategyCode	int,"
        #     "Comments	string,"
        #     "LastUpdatedDate	timestamp,"
        #     "CreatedDate	timestamp,"
        #     "CreatedBy	string,"
        #     "AssigneeUserCode	string,"
        #     "AssigneeUser	string,"
        #     "AssigneePortfolioGroupCode	string,"
        #     "AssigneePortfolioGroup	string,"
        #     "IntegrationCode	string,"
        #     "CommunicationChannelCode	string,"
        #     "CommunicationChannel	string,"
        #     "FlowSegmentCode	string,"
        #     "FlowSegment	string,"
        #     "UserCreatedQualcoGroupCode	string,"
        #     "ContractSigningMethodCode	string,"
        #     "ContractSigningMethod	string,"
        #     "ManagedByPortfolioGroupCode	string,"
        #     "InterbankFlag	boolean,"
        #     "OcwFlag	boolean,"
        #     "ContactSigningPlaceTypeCode	int,"
        #     "ContactSigningPlaceType	string,"
        #     "ReportingApplicationStatusCode	int,"
        #     "ReportingApplicationStatus	string,"
        #     "ClientResponseCode	int,"
        #     "ClientResponse	string,"
        #     "OcwSimulationVote	int,"
        #     "StartedDate	timestamp,"
        #     "SubmittedDate	timestamp,"
        #     "QualcoAppPreApprovedDate	timestamp,"
        #     "QualcoAppDisbursedDate	timestamp,"
        #     "QualcoAppFinalStepDate	timestamp,"
        #     "UserCreatedChannelCode	string,"
        #     "UserUnit	string,"
        #     "UserCreatedGroupCode	string,"
        #     "UserCreatedGroup	string,"
        #     "ApplicationImplementationStatusCode	string,"
        #     "ApplicationImplementationStatus	string,"
        #     "ApplicationActivationDate	date,"
        #     "ContactSigningPlaceCode	int,"
        #     "PolicyOverrideFlag	boolean,"
        #     "InstallmentNumber int,"
        #     "InstallmentAmount decimal(18,4),"
        #     "InstallmentDate timestamp,"
        #     "InstallmentCapitalAmount decimal(18,4),"
        #     "InstallmentChargeAmount decimal(18,4),"
        #     "InstallmentN128Amount decimal(18,4),"
        #     "InstallmentTotalChargeAmount decimal(18,4),"
        #     "InstallmentRemainingCapitalAmount decimal(18,4),"
        #     "InstallmentNPVAmount decimal(18,4),"
        #     "InstallmentNPVCalculationDate timestamp,"
        #     "InstallementStatusFlag boolean,"
        #     "IsPrimaryRowFlag int,"
        #     "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Application"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "fldt",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")

        # ================== application ==================

        application_extra_filters = None

        if application_extra_filters is not None:
            application_extra_filters.append(date_filter)
            application_final_filters = " AND ".join(application_extra_filters)
        else:
            application_final_filters = date_filter

        application_selected_fields = [
            "ApplicationId",
            "ApplicationKey",
            "PortfolioName",
            "id",
            "ApplicationStatus",
            "DateCreated",
            "ProcessingUserName",
            "DateUpdated",
            "UserCreated",
            "AssigneeUserId",
            "AssigneeUserName",
            "AssigneePortfolioGroupCode",
            "AssigneePortfolioGroupName",
            "IntegrationCode",
            "CommunicationChannelCode",
            "CommunicationChannelDescription",
            "FlowSegment",
            "FlowSegmentDescription",
            "UserCreatedQualcoGroupCode",
            "ContractSigningMethodCode",
            "ContractSigningMethodDescription",
            "ManagedByPortfolioGroupCode",
            "InterbankFlag",
            "OcwFlag",
            "ContactSigningPlaceTypeId",
            "ContactSigningPlaceTypeName",
            "UserCreatedQualcoChannelCode",
            "UserUnitCode",
            "UserCreatedGroupCode",
            "UserCreatedGroupName",
            "ContactSigningPlaceId",
            "ApplicationStatusDescription",
            "PolicyOverride",
            "customercode",
            "businessdate",
        ]

        application = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_application",
            application_final_filters,
            application_selected_fields,
        )

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "customercode",
            "vatnumber",
            "groupcode",
            "applicationdate",
            "IntentionDate",
            "CompletionTargetDate",
            "positiondate",
            "applicationid",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_customer",
            customer_final_filters,
            customer_selected_fields,
        )
        customer = customer.withColumnRenamed("customercode", "customercode_c")

        # ================== application_metadata ==================

        application_metadata_extra_filters = None

        if application_metadata_extra_filters is not None:
            application_metadata_extra_filters.append(date_filter)
            application_metadata_final_filters = " AND ".join(
                application_metadata_extra_filters
            )
        else:
            application_metadata_final_filters = date_filter

        application_metadata_selected_fields = [
            "QualcoAppApprovedDate",
            "QualcoAppRejectedDate",
            "QualcoAppCancelledDate",
            "ApplicationStatus",
            "ApplicationStatusDescription",
            "ClientResponseId",
            "ClientResponseDescription",
            "OcwSimulationVoteId",
            "QualcoAppStartedDate",
            "QualcoAppSubmittedDate",
            "QualcoAppPreApprovedDate",
            "QualcoAppDisbursedDate",
            "QualcoAppFinalStepDate",
            "ApplicationImplementationStatus",
            "ApplicationActivationDate",
            "IsActive",
            "SourceApplicationKey",
        ]

        application_metadata = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_applicationmetadata",
            application_metadata_final_filters,
            application_metadata_selected_fields,
        )
        application_metadata = application_metadata.withColumnRenamed(
            "ApplicationStatus", "ApplicationStatus_application_metadata"
        )

        # ================== application_fldt_df ==================

        application_fldt_primary_key_fields = [
            application.ApplicationId,
            lit("fldt"),
        ]

        window_spec = Window.partitionBy("applicationid", "customercode_c").orderBy(
            customer["positiondate"].desc()
        )

        cust_subquery = (
            customer.select(
                customer["applicationid"],
                customer["customercode_c"],
                customer["positiondate"],
                customer["vatnumber"],
                customer["groupcode"],
                customer["applicationdate"],
                customer["CompletionTargetDate"],
                customer["IntentionDate"],
                lit(1).alias("rn"),
            )
            .withColumn("rn", row_number().over(window_spec))
            .filter(col("rn") == 1)
        )

        application_fldt_df = (
            application.join(
                application_metadata,
                (
                    application_metadata["SourceApplicationKey"]
                    == application["ApplicationKey"]
                )
                & (application_metadata["IsActive"] == 1),
                "left",
            )
            .join(
                cust_subquery,
                (cust_subquery["applicationid"] == application["id"])
                & (cust_subquery["rn"] == 1),
                "inner",
            )
            .withColumn(
                "ApplicationIdKey",
                sha2(concat_ws(";", *application_fldt_primary_key_fields), 256),
            )
        )

        # ================== Selection of Final Columns ==================

        application_fldt_df = application_fldt_df.select(
            application_fldt_df.ApplicationIdKey.cast("string").alias("ApplicationId"),
            lit("fldt").alias("SourceSystem"),
            substring(col("customercode"), 1, 3).cast("string").alias("PortfolioKey"),
            application_fldt_df.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            application.ApplicationKey.alias("ApplicationKey"),
            lit(None).alias("ApplicationStrategyKey"),
            lit(None).alias("ApplicationScheduleKey"),
            lit(None).alias("ExposureKey"),
            lit(None).alias("AgreementKey"),
            application_fldt_df.customercode_c.alias("CDI"),
            application_fldt_df.vatnumber.cast("string").alias("TaxRegistrationNumber"),
            application_fldt_df.groupcode.cast("string").alias("GroupCode"),
            lit(None).alias("BusinessPlanKey"),
            lit(None).alias("BusinessPlanSolutionTypeCode"),
            lit(None).alias("BusinessPlanSolutionType"),
            lit(None).alias("BusinessPlanSolutionStatusCode"),
            lit(None).alias("BusinessPlanSolutionStatus"),
            lit(None).alias("BusinessPlanComments"),
            lit(None).alias("BusinessPlanCreatedDate"),
            lit(None).alias("BusinessPlanCreatedBy"),
            lit(None).alias("BusinessPlanLastUpdatedDate"),
            lit(None).alias("BusinessPlanLastUpdatedBy"),
            application_fldt_df.customercode_c.alias("PartyKey"),
            lit("RestructuringsFLDT").alias("ApplicationTypeCode"),
            lit("RestructuringsFLDT").alias("ApplicationType"),
            application_fldt_df.ApplicationStatus.cast("string").alias(
                "ApplicationStatusCode"
            ),
            application.ApplicationStatusDescription.cast("string").alias(
                "ApplicationStatus"
            ),
            application_fldt_df.applicationdate.alias("ApplicationDate"),
            application_fldt_df.IntentionDate.cast("date").alias("IntentionDate"),
            application_fldt_df.QualcoAppApprovedDate.alias("ApprovedDate"),
            application_fldt_df.QualcoAppRejectedDate.alias("RejectedDate"),
            application_fldt_df.QualcoAppCancelledDate.alias("CancelledDate"),
            application_fldt_df.CompletionTargetDate.cast("string").alias(
                "ImplementationDate"
            ),
            lit(None).alias("BeginDate"),
            lit(None).alias("ProposedTypeCode"),
            lit(None).alias("ProposedType"),
            lit(None).alias("ConfidenceTypeCode"),
            lit(None).alias("ConfidenceType"),
            lit(None).alias("Installment"),
            application_fldt_df.ProcessingUserName.alias("PropositionUser"),
            lit(None).alias("Tenor"),
            lit(None).alias("TimeToAuction"),
            lit(None).alias("TimeToRecovery"),
            lit(None).alias("TimeToSell"),
            lit(None).alias("PrepaymentDate"),
            lit(None).alias("AuctionPrice"),
            lit(None).alias("BalloonAmount"),
            lit(None).alias("PrepaymentAmount"),
            lit(None).alias("BusinessPlanStrategySolutionTypeCode"),
            lit(None).alias("BusinessPlanStrategySolutionType"),
            lit(None).alias("Interest"),
            lit(None).alias("InterestPaymentPeriodicity"),
            lit(None).alias("NetRecoveryAmount"),
            lit(None).alias("BusinessPlanStrategyCode"),
            lit(None).alias("Comments"),
            application_fldt_df.DateUpdated.alias("LastUpdatedDate"),
            application_fldt_df.DateCreated.cast("timestamp").alias("CreatedDate"),
            application_fldt_df.UserCreated.cast("string").alias("CreatedBy"),
            application_fldt_df.AssigneeUserId.alias("AssigneeUserCode"),
            application_fldt_df.AssigneeUserName.alias("AssigneeUser"),
            application_fldt_df.AssigneePortfolioGroupCode.alias(
                "AssigneePortfolioGroupCode"
            ),
            application_fldt_df.AssigneePortfolioGroupName.alias(
                "AssigneePortfolioGroup"
            ),
            application_fldt_df.IntegrationCode.alias("IntegrationCode"),
            application_fldt_df.CommunicationChannelCode.alias(
                "CommunicationChannelCode"
            ),
            application_fldt_df.CommunicationChannelDescription.alias(
                "CommunicationChannel"
            ),
            application_fldt_df.FlowSegment.alias("FlowSegmentCode"),
            application_fldt_df.FlowSegmentDescription.alias("FlowSegment"),
            application_fldt_df.UserCreatedQualcoGroupCode.alias(
                "UserCreatedQualcoGroupCode"
            ),
            application_fldt_df.ContractSigningMethodCode.alias(
                "ContractSigningMethodCode"
            ),
            application_fldt_df.ContractSigningMethodDescription.alias(
                "ContractSigningMethod"
            ),
            application_fldt_df.ManagedByPortfolioGroupCode.alias(
                "ManagedByPortfolioGroupCode"
            ),
            application_fldt_df.InterbankFlag.alias("InterbankFlag"),
            application_fldt_df.OcwFlag.alias("OcwFlag"),
            application_fldt_df.ContactSigningPlaceTypeId.alias(
                "ContactSigningPlaceTypeCode"
            ),
            application_fldt_df.ContactSigningPlaceTypeName.alias(
                "ContactSigningPlaceType"
            ),
            application_fldt_df.ApplicationStatus_application_metadata.alias(
                "ReportingApplicationStatusCode"
            ),
            application_metadata.ApplicationStatusDescription.alias(
                "ReportingApplicationStatus"
            ),
            application_fldt_df.ClientResponseId.alias("ClientResponseCode"),
            application_fldt_df.ClientResponseDescription.alias("ClientResponse"),
            application_fldt_df.OcwSimulationVoteId.alias("OcwSimulationVote"),
            application_fldt_df.QualcoAppStartedDate.alias("StartedDate"),
            application_fldt_df.QualcoAppSubmittedDate.alias("SubmittedDate"),
            application_fldt_df.QualcoAppPreApprovedDate.alias(
                "QualcoAppPreApprovedDate"
            ),
            application_fldt_df.QualcoAppDisbursedDate.alias("QualcoAppDisbursedDate"),
            application_fldt_df.QualcoAppFinalStepDate.alias("QualcoAppFinalStepDate"),
            application_fldt_df.UserCreatedQualcoChannelCode.alias(
                "UserCreatedChannelCode"
            ),
            application_fldt_df.UserUnitCode.alias("UserUnit"),
            application_fldt_df.UserCreatedGroupCode.alias("UserCreatedGroupCode"),
            application_fldt_df.UserCreatedGroupName.alias("UserCreatedGroup"),
            application_fldt_df.ApplicationImplementationStatus.alias(
                "ApplicationImplementationStatusCode"
            ),
            application_fldt_df.ApplicationImplementationStatus.alias(
                "ApplicationImplementationStatus"
            ),
            application_fldt_df.ApplicationActivationDate.alias(
                "ApplicationActivationDate"
            ),
            application_fldt_df.ContactSigningPlaceId.alias("ContactSigningPlaceCode"),
            application_fldt_df.PolicyOverride.alias("PolicyOverrideFlag"),
            lit(None).alias("InstallmentNumber"),
            lit(None).alias("InstallmentAmount"),
            lit(None).alias("InstallmentDate"),
            lit(None).alias("InstallmentCapitalAmount"),
            lit(None).alias("InstallmentChargeAmount"),
            lit(None).alias("InstallmentN128Amount"),
            lit(None).alias("InstallmentTotalChargeAmount"),
            lit(None).alias("InstallmentRemainingCapitalAmount"),
            lit(None).alias("InstallmentNPVAmount"),
            lit(None).alias("InstallmentNPVCalculationDate"),
            lit(None).alias("InstallementStatusFlag"),
            lit(None).alias("IsPrimaryRowFlag"),
            application_fldt_df.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        application_fldt_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_application_fldt(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
)
